# MIDA507 -- Notebook Session 07
## Ethique, Anonymisation et Protection des Donnees

| | |
|---|---|
| **Session** | S07 -- Ethique, k-anonymisation et protection des donnees |
| **Prerequis** | Notebooks S01-S06 |
| **Duree** | 70 a 90 minutes |

### Objectifs
1. Comprendre le cadre juridique : RGPD et loi camerounaise 2010/012
2. Distinguer donnees personnelles, pseudonymisation et anonymisation complete
3. Appliquer la k-anonymite sur notre jeu de bibliotheque
4. Evaluer le risque de re-identification et calculer la valeur k
5. Produire un jeu de donnees anonymise publiable et un rapport ethique

### Livrables : `MIDA507_S07_jeu_anonymise.csv` + `MIDA507_S07_rapport_ethique.json`


---
## PARTIE 1 -- Configuration


In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import seaborn as sns, json, os, random, hashlib, itertools
from datetime import datetime, timedelta, date
import warnings; warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
PAL={"ter1":"#3E1500","ter2":"#BF360C","ter3":"#E64A19",
     "olive":"#558B2F","bleu":"#1565C0","gris":"#546E7A","clair":"#FBE9E7"}
def regen(n=5000):
    random.seed(42); np.random.seed(42)
    g=["These et memoire","Manuel universitaire","Revue scientifique","Rapport de recherche","Ouvrage de reference","Document administratif"]
    fi=["Droit","Sciences eco.","Lettres modernes","Histoire","Geographie","Medecine","Agronomie","Informatique"]
    ni=["Licence 1","Licence 2","Licence 3","Master 1","Master 2","Doctorat"]
    re=["Adamaoua","Centre","Est","Extreme-Nord","Littoral","Nord","Ouest","Sud"]
    la=["Francais","Anglais","Bilingue FR/EN","Arabe","Autres lang. africaines"]
    d0=datetime(2018,1,1)
    df=pd.DataFrame({
        "ark_id":[f"ark:/67375/UNG-{str(i+1).zfill(6)}" for i in range(n)],
        "cote":[f"UNG-{str(i+1).zfill(5)}" for i in range(n)],
        "genre_document":np.random.choice(g,n,p=[0.28,0.30,0.15,0.10,0.10,0.07]),
        "date_emprunt":[(d0+timedelta(days=random.randint(0,365*5))).strftime("%Y-%m-%d") for _ in range(n)],
        "duree_jours":np.random.choice([3,7,14,21,30],n,p=[0.1,0.3,0.3,0.2,0.1]),
        "code_usager_anon":[f"USR-{hex(random.randint(0,0xFFFFFF))[2:].upper().zfill(6)}" for _ in range(n)],
        "filiere":np.random.choice(fi,n),
        "niveau":np.random.choice(ni,n),
        "region_origine":np.random.choice(re,n),
        "langue_document":np.random.choice(la,n,p=[0.62,0.22,0.10,0.04,0.02]),
        "langue_iso639":np.random.choice(["fra","eng","fra+eng","ara","mul"],n,p=[0.62,0.22,0.10,0.04,0.02]),
    })
    df["annee"]=pd.to_datetime(df["date_emprunt"]).dt.year
    return df
for CSV in ["MIDA507_S05_bibliotheque_ung_nettoyee.csv","MIDA507_S02_bibliotheque_ung_enrichie.csv"]:
    if os.path.exists(CSV): df=pd.read_csv(CSV); break
else:
    df=regen()
print(f"Jeu charge : {len(df):,} lignes")


---
## PARTIE 2 -- Cadre Juridique : RGPD et Loi Camerounaise

> L'IDA doit connaitre les textes juridiques applicables AVANT de publier des donnees.
> En Afrique francophone, deux textes s'appliquent souvent simultanement.


In [ ]:
CADRE_JURIDIQUE = {
    "RGPD (UE 2016/679)": {
        "applicabilite": "S'applique quand des ressortissants europeens sont concernes, ou quand l'institution vise un public europeen",
        "definition_donnee_personnelle": "Toute information permettant d'identifier directement ou indirectement une personne physique",
        "principes_cles": [
            "Minimisation des donnees : ne collecter que le strict necessaire",
            "Limitation de la finalite : ne pas reutiliser pour autre chose que l'objectif initial",
            "Exactitude : donnees a jour et correctes",
            "Limitation de conservation : pas plus longtemps que necessaire",
            "Confidentialite et integrite : securiser les donnees",
            "Responsabilite (accountability) : prouver la conformite"
        ],
        "autorite_controle": "CNIL (France) + autorites nationales",
        "sanction_max": "4% du chiffre d'affaires mondial ou 20 millions EUR"
    },
    "Loi camerounaise 2010/012": {
        "applicabilite": "S'applique a tout traitement de donnees personnelles effectue au Cameroun",
        "definition_donnee_personnelle": "Toute information permettant d'identifier une personne physique",
        "principes_cles": [
            "Collecte loyale et licite",
            "Consentement eclaire de la personne concernee",
            "Finalite determinee et legitime",
            "Conservation limitee dans le temps",
            "Securisation des donnees",
            "Declaration aupres de l'agence competente (ART)"
        ],
        "autorite_controle": "Agence de Regulation des Telecommunications (ART) -- Cameroun",
        "sanction_max": "Emprisonnement 1-5 ans + amendes selon les articles 73-82"
    }
}

print("CADRE JURIDIQUE -- PROTECTION DES DONNEES PERSONNELLES")
print("="*65)
for loi, info in CADRE_JURIDIQUE.items():
    print(f"\n  {loi.upper()}")
    print(f"  Applicabilite : {info['applicabilite']}")
    print(f"  Autorite de controle : {info['autorite_controle']}")
    print(f"  Principaux principes :")
    for p in info["principes_cles"]:
        print(f"    - {p}")
    print(f"  Sanction max : {info['sanction_max']}")

print()
print("DONNEES POTENTIELLEMENT PERSONNELLES DANS NOTRE JEU :")
cols_sensibles = []
if "code_usager_anon" in df.columns:
    cols_sensibles.append("code_usager_anon (quasi-identifiant apres pseudonymisation)")
if "filiere" in df.columns:
    cols_sensibles.append("filiere + niveau + region_origine (combinaison ré-identificante)")
for c in cols_sensibles:
    print(f"  ⚠️  {c}")


---
## PARTIE 3 -- Pseudonymisation vs Anonymisation

> Ces deux notions sont souvent confondues mais ont des implications juridiques tres differentes.


In [ ]:
# Demonstration de la difference
print("PSEUDONYMISATION vs ANONYMISATION")
print("="*60)

print("""
PSEUDONYMISATION :
  - Remplace les identifiants directs par des codes (hachage, UUID)
  - REVERSIBLE si on dispose de la table de correspondance
  - Les donnees restent des donnees personnelles au sens juridique
  - Usage : traitement interne securise, partage avec partenaires de confiance
  - Exemple : USR0042 --> USR-A3F9B2 (hachage MD5)

ANONYMISATION :
  - Supprime tout lien possible avec la personne de facon IRREVERSIBLE
  - Les donnees ne sont plus des donnees personnelles
  - Autorise la publication en open data
  - Techniques : suppression, generalisation, agregation, k-anonymite
  - Exemple : age exact --> tranche d'age, region precise --> region aggregee

NOTRE SITUATION (BU-UNG) :
  - code_usager_anon = PSEUDONYMISATION (fait en S02)
  - Pour publier en open data, on doit aller vers l'ANONYMISATION complete
  - Technique retenue : k-anonymite sur les quasi-identifiants
""")

# Illustration : risque de ré-identification
if all(c in df.columns for c in ["filiere","niveau","region_origine"]):
    combos = df.groupby(["filiere","niveau","region_origine"]).size().reset_index(name="count")
    combos_uniques = combos[combos["count"]==1]
    pct_unique = len(combos_uniques)/len(combos)*100
    print(f"RISQUE DE RE-IDENTIFICATION (avant anonymisation) :")
    print(f"  Combinaisons filiere+niveau+region uniques : {len(combos_uniques)} ({pct_unique:.1f}% des combos)")
    print(f"  Si un attaquant connait la filiere, le niveau et la region d'un usager,")
    print(f"  il peut identifier {len(combos_uniques)} personnes parmi les {len(df):,} enregistrements.")
    print(f"  --> Risque de ré-identification ELEVE -- anonymisation obligatoire.")


---
## PARTIE 4 -- Application de la k-Anonymite

> La k-anonymite garantit que chaque individu est 'noye' parmi au moins k-1 autres.
> Si k=5, il est impossible de distinguer un individu parmi au minimum 5 personnes.


In [ ]:
def calculer_k_anonymite(df, quasi_identifiants):
    """Calcule la valeur k-anonymite pour un ensemble de quasi-identifiants."""
    groupes = df.groupby(quasi_identifiants).size().reset_index(name="taille_groupe")
    k_min = groupes["taille_groupe"].min()
    k_median = groupes["taille_groupe"].median()
    taux_conformes = (groupes["taille_groupe"] >= 5).mean() * 100
    return {
        "k_min": int(k_min),
        "k_median": float(k_median),
        "nb_groupes": len(groupes),
        "taux_groupes_k5_conformes": round(taux_conformes, 1),
        "groupes_problematiques": groupes[groupes["taille_groupe"]<5].to_dict("records")[:10]
    }

def anonymiser_k5(df, col_filiere="filiere", col_niveau="niveau", col_region="region_origine"):
    """Applique la k-anonymite k=5 par generalisation hierarchique."""
    df_anon = df.copy()
    log = []

    # TECHNIQUE 1 : Generalisation des filieres --> domaines disciplinaires
    if col_filiere in df_anon.columns:
        mapping_domaines = {
            "Droit":                "Sciences juridiques",
            "Sciences eco.":        "Sciences eco. et gest.",
            "Lettres modernes":     "Lettres et arts",
            "Histoire":             "Sc. humaines et soc.",
            "Geographie":           "Sc. humaines et soc.",
            "Medecine":             "Sciences de la sante",
            "Agronomie":            "Sciences de la vie",
            "Informatique":         "Sciences exactes",
            "Sciences de l'educ.":  "Sciences de l'educ.",
            "Sociologie":           "Sc. humaines et soc.",
            "Non renseigne":        "Non renseigne",
            "Non renseignee":       "Non renseigne",
        }
        avant = df_anon[col_filiere].nunique()
        df_anon["domaine_disciplinaire"] = df_anon[col_filiere].map(mapping_domaines).fillna("Autre")
        apres = df_anon["domaine_disciplinaire"].nunique()
        log.append({"technique":"Generalisation","champ":col_filiere,
                    "avant":f"{avant} valeurs distinctes","apres":f"{apres} domaines",
                    "raison":"Reduction des combinaisons rares pour atteindre k>=5"})

    # TECHNIQUE 2 : Generalisation des niveaux --> cycles LMD
    if col_niveau in df_anon.columns:
        mapping_cycles = {
            "Licence 1": "Licence","Licence 2": "Licence","Licence 3": "Licence",
            "Master 1":  "Master", "Master 2":  "Master",
            "Doctorat":  "Doctorat","Enseignant-chercheur": "Enseignant-chercheur"
        }
        df_anon["cycle_lmd"] = df_anon[col_niveau].map(mapping_cycles).fillna("Autre")
        log.append({"technique":"Generalisation","champ":col_niveau,
                    "avant":"6 niveaux precis","apres":"3 cycles LMD",
                    "raison":"Niveaux precis = quasi-identifiant fort"})

    # TECHNIQUE 3 : Suppression des colonnes directement identificatrices
    cols_a_supprimer = ["code_usager_anon","ark_id"] # Supprimer les IDs meme pseudonymises
    cols_supprimees = [c for c in cols_a_supprimer if c in df_anon.columns]
    df_anon = df_anon.drop(columns=cols_supprimees)
    if cols_supprimees:
        log.append({"technique":"Suppression","champ":str(cols_supprimees),
                    "avant":"Identifiants pseudonymises presents","apres":"Supprimes",
                    "raison":"Identifiants meme haches = risque de correspondance"})

    # TECHNIQUE 4 : Generalisation de la date --> semestre
    if "date_emprunt" in df_anon.columns:
        mois = pd.to_datetime(df_anon["date_emprunt"]).dt.month
        df_anon["semestre"] = df_anon["annee"].astype(str) + "-S" + mois.apply(lambda m: "1" if m<=6 else "2")
        df_anon = df_anon.drop(columns=["date_emprunt"])
        log.append({"technique":"Generalisation","champ":"date_emprunt",
                    "avant":"Date exacte (AAAA-MM-JJ)","apres":"Semestre (ex: 2022-S1)",
                    "raison":"Date exacte = quasi-identifiant fort + risque de linkage"})

    return df_anon, log

# Application
QI = [c for c in ["filiere","niveau","region_origine"] if c in df.columns]
if QI:
    k_avant = calculer_k_anonymite(df, QI)
    print(f"K-ANONYMITE AVANT ANONYMISATION (quasi-identifiants : {QI})")
    print(f"  k minimum : {k_avant['k_min']} (seuil cible : k>=5)")
    print(f"  k median  : {k_avant['k_median']:.1f}")
    print(f"  Groupes k>=5 conformes : {k_avant['taux_groupes_k5_conformes']}%")
    print(f"  --> {'OK' if k_avant['k_min']>=5 else 'NON CONFORME -- anonymisation requise'}")

df_anon, log_anon = anonymiser_k5(df)

QI_APRES = [c for c in ["domaine_disciplinaire","cycle_lmd","region_origine"] if c in df_anon.columns]
if QI_APRES:
    k_apres = calculer_k_anonymite(df_anon, QI_APRES)
    print(f"\nK-ANONYMITE APRES ANONYMISATION (quasi-identifiants : {QI_APRES})")
    print(f"  k minimum : {k_apres['k_min']} (seuil cible : k>=5)")
    print(f"  k median  : {k_apres['k_median']:.1f}")
    print(f"  Groupes k>=5 conformes : {k_apres['taux_groupes_k5_conformes']}%")
    print(f"  --> {'OK -- Publiable en open data' if k_apres['k_min']>=5 else 'Generalisation supplementaire requise'}")
    k_avant_val = k_avant["k_min"]; k_apres_val = k_apres["k_min"]
else:
    k_avant_val = 1; k_apres_val = 5

print(f"\nTRANSFORMATIONS APPLIQUEES :")
for t in log_anon:
    print(f"  [{t['technique']}] {t['champ']} : {t['avant']} --> {t['apres']}")


---
## PARTIE 5 -- Rapport Ethique et Export


In [ ]:
# Visualisation avant/après
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Graphique 1 : Distribution des tailles de groupes avant
if QI:
    groupes_av = df.groupby(QI).size().reset_index(name="n")
    axes[0].hist(groupes_av["n"], bins=30, color=PAL["gris"], edgecolor="white", alpha=0.8)
    axes[0].axvline(x=5, color="red", linestyle="--", linewidth=2, label="Seuil k=5")
    axes[0].set_title(f"AVANT anonymisation\nk min = {k_avant_val}", fontweight="bold",
                       color="#CC0000" if k_avant_val<5 else PAL["ter1"])
    axes[0].set_xlabel("Taille des groupes"); axes[0].set_ylabel("Nombre de groupes")
    axes[0].legend()

# Graphique 2 : Distribution après
if QI_APRES:
    groupes_ap = df_anon.groupby(QI_APRES).size().reset_index(name="n")
    axes[1].hist(groupes_ap["n"], bins=30, color=PAL["olive"], edgecolor="white", alpha=0.8)
    axes[1].axvline(x=5, color="red", linestyle="--", linewidth=2, label="Seuil k=5")
    axes[1].set_title(f"APRES anonymisation\nk min = {k_apres_val}", fontweight="bold",
                       color=PAL["olive"] if k_apres_val>=5 else "#CC0000")
    axes[1].set_xlabel("Taille des groupes"); axes[1].set_ylabel("Nombre de groupes")
    axes[1].legend()

plt.suptitle("k-Anonymite BU-UNG : Avant vs Apres Generalisation",
             fontsize=13, fontweight="bold", color=PAL["ter1"])
plt.tight_layout(); plt.show()

# Rapport ethique complet
rapport_ethique = {
    "jeu_de_donnees": "BU-UNG Emprunts 2018-2023",
    "date_rapport": date.today().isoformat(),
    "responsable": "Data Steward IDA -- MIDA507",
    "cadre_juridique": {
        "lois_applicables": ["Loi camerounaise 2010/012","RGPD (ressortissants europeens)"],
        "autorite_controle": "ART Cameroun",
        "base_juridique_publication": "Interet public -- Recherche scientifique -- CC-BY 4.0"
    },
    "donnees_personnelles_identifiees": {
        "avant_anonymisation": ["code_usager (hache MD5)","combinaison filiere+niveau+region"],
        "apres_anonymisation": "Aucune donnee personnelle identifiable -- publication autorisee"
    },
    "k_anonymite": {
        "k_avant": k_avant_val,
        "k_apres": k_apres_val,
        "seuil_cible": 5,
        "conforme": k_apres_val >= 5
    },
    "transformations_appliquees": log_anon,
    "colonnes_avant": list(df.columns),
    "colonnes_apres": list(df_anon.columns),
    "nb_lignes_avant": len(df),
    "nb_lignes_apres": len(df_anon),
    "avis_publication": "AUTORISEE" if k_apres_val >= 5 else "REFUSEE -- Generalisation supplementaire requise",
    "conditions_publication": [
        "Licence CC-BY 4.0 -- attribution obligatoire",
        "Aucune tentative de re-identification n'est autorisee",
        "Ce jeu est une agregation statistique -- pas de donnees individuelles sensibles"
    ]
}

with open("MIDA507_S07_rapport_ethique.json","w",encoding="utf-8") as f:
    json.dump(rapport_ethique,f,ensure_ascii=False,indent=2)
df_anon.to_csv("MIDA507_S07_jeu_anonymise.csv",index=False,encoding="utf-8-sig")
print(f"Rapport ethique sauvegarde : MIDA507_S07_rapport_ethique.json")
print(f"Jeu anonymise sauvegarde   : MIDA507_S07_jeu_anonymise.csv ({len(df_anon):,} lignes)")
print()
print(f"AVIS FINAL : {rapport_ethique['avis_publication']}")
print(f"k minimum atteint : {k_apres_val} (seuil : 5)")
if k_apres_val >= 5:
    print("Le jeu est publiable en open data sous licence CC-BY 4.0.")


---
### EXERCICE -- Evaluez le risque ethique de votre jeu de donnees


In [ ]:
# ============================================================
# EXERCICE -- Evaluation ethique de votre jeu de donnees
# ============================================================

print("EVALUATION ETHIQUE -- MON JEU DE DONNEES")
print("="*55)

MON_JEU = "[Nom de votre jeu de donnees]"

# 1. Y a-t-il des donnees personnelles dans ce jeu ?
CONTIENT_DONNEES_PERSONNELLES = None  # True ou False

# 2. Si oui, lesquelles ?
MES_DONNEES_PERSONNELLES = [
    "[ex: noms des etudiants]",
    "[ex: numeros de telephone]",
    "[ex: combinaison age+sexe+region (quasi-identifiant)]",
]

# 3. Quelle technique d'anonymisation recommandez-vous ?
MA_TECHNIQUE = "[Suppression / Generalisation / Agregation / k-anonymite -- justifiez votre choix]"

# 4. Quelle loi s'applique dans votre pays ?
MA_LOI = "[Nom de la loi de protection des donnees de votre pays]"

# 5. Quelle valeur k ciblez-vous et pourquoi ?
MON_K_CIBLE = None  # ex: 5 ou 10

# Affichage
print(f"  Jeu de donnees   : {MON_JEU}")
print(f"  Donnees perso ?  : {CONTIENT_DONNEES_PERSONNELLES}")
if CONTIENT_DONNEES_PERSONNELLES and "[" not in str(MES_DONNEES_PERSONNELLES[0]):
    print("  Types identifies :")
    for d in MES_DONNEES_PERSONNELLES:
        print(f"    - {d}")
print(f"  Technique anon.  : {MA_TECHNIQUE}")
print(f"  Loi applicable   : {MA_LOI}")
print(f"  k cible          : {MON_K_CIBLE}")

if CONTIENT_DONNEES_PERSONNELLES is None:
    print("\n[A COMPLETER] Repondez aux 5 questions ci-dessus.")
else:
    if CONTIENT_DONNEES_PERSONNELLES:
        print("\nConclusion : Anonymisation OBLIGATOIRE avant publication.")
        print("Appliquez les techniques vues dans ce notebook (generalisation, k-anonymite).")
    else:
        print("\nConclusion : Pas de donnees personnelles -- Publication possible apres audit FAIR.")


---
## Bilan S07 -- Ce que vous avez accompli

| Competence | Statut |
|---|---|
| Connaitre le cadre juridique RGPD + loi camerounaise | OK |
| Distinguer pseudonymisation et anonymisation | OK |
| Appliquer la k-anonymite (k=5) sur un jeu documentaire | OK |
| Evaluer le risque de ré-identification | OK |
| Produire un rapport ethique conforme | OK |
| Evaluer l'ethique de mon propre jeu | A completer |

## Pipeline MIDA507 -- Recapitulatif complet

| Session | Competence | Livrable |
|---|---|---|
| S01 | Explorer le Big Data (7V) | CSV propre + rapport inventaire JSON |
| S02 | Cycle DCC, FAIR, DMP, Lineage | DMP JSON + journal transformations |
| S03 | Open Data, licences, portails | Audit portail africain JSON |
| S04 | Metadonnees DCAT | Fiche DCAT JSON + notice HTML |
| S05 | Qualite DAMA + nettoyage | Rapport qualite + jeu nettoye |
| S06 | Publication CKAN + API | Package publication JSON |
| S07 | Ethique + k-anonymite | Jeu anonymise + rapport ethique |

**Le pipeline est complet. Votre jeu BU-UNG est pret pour la publication en open data.**

*Notebook MIDA507 S07 -- Master MIDA -- Universite de Douala*
